#AlphaZero Chess Reinforcement Learning (AlphaZeroNet V1) on Kaggle GPU

Welcome to this self-contained Jupyter Notebook designed to train **AlphaZeroNet V1** using **Monte Carlo Tree Search (MCTS)** and **Self-Play Reinforcement Learning**. This notebook is fully optimized to run out-of-the-box on Kaggle's free **Tesla T4 GPU** environment, featuring high-speed parallel self-play and batched neural network inference.

---

##1. Theoretical Foundation & Model Architecture

AlphaZero, introduced by Google DeepMind, represents a paradigm shift in artificial intelligence. Instead of relying on human-crafted heuristics or handcrafted evaluation functions (like Stockfish), AlphaZero learns chess tabula rasa (from scratch) solely through **self-play reinforcement learning** guided by Monte Carlo Tree Search.

###1.1. State Representation (Input Tensor)
A chess board is represented as an $18 \times 8 \times 8$ float32 tensor $s$, consisting of 18 binary layers:
*   **Planes 0 to 5:** White pieces (Pawn, Knight, Bishop, Rook, Queen, King). An element is `1.0` if a piece is present, else `0.0`.
*   **Planes 6 to 11:** Black pieces (Pawn, Knight, Bishop, Rook, Queen, King).
*   **Plane 12:** Whose turn is it (entire grid is `1.0` if White's turn, `0.0` if Black's turn).
*   **Planes 13 to 16:** Castling rights (White Kingside, White Queenside, Black Kingside, Black Queenside - filled with `1.0` if allowed).
*   **Plane 17:** En-passant square (set to `1.0` on the active en-passant destination square).

###1.2. Monte Carlo Tree Search (MCTS) & PUCT selection
AlphaZero maintains a dynamic localized search tree where each edge $(s, a)$ stores a set of metrics:
*   $N(s, a)$: Visit count.
*   $W(s, a)$: Total action value accumulated from leaf node backpropagations.
*   $Q(s, a)$: Mean action-value, calculated dynamically as:

    $$
    Q(s, a) = \frac{W(s, a)}{N(s, a)}
    $$
During tree traversal, MCTS selects the child action $a^*$ that maximizes the **PUCT (Predictor Upper Confidence Trees)** formula:
$$a^* = \arg\max_{a \in \text{Legals}(s)} \left( Q(s, a) + C_{\text{puct}} \cdot P(s, a) \cdot \frac{\sqrt{\sum_{b} N(s, b)}}{1 + N(s, a)} \right)$$
where:
*   $P(s, a)$ is the prior probability of selecting move $a$ predicted by the neural network's **Policy Head**.
*   $\sum_b N(s, b)$ represents the total visit count of the parent node $s$.
*   $C_{\text{puct}}$ is a scaling hyperparameter controlling exploration vs. exploitation.

###1.3. Opening Exploration & Dirichlet Noise
To ensure the self-play agent searches diverse trajectories, **Dirichlet Noise** is injected directly into the root node's prior probabilities:
$$P(s, a) = (1 - \epsilon) P_0(s, a) + \epsilon \cdot \eta_a$$
where $\eta \sim \text{Dir}(\alpha)$ represents a sample from a Dirichlet distribution (with $\alpha = 0.3$ standard for chess) and $\epsilon = 0.25$ is the noise blend factor.

Once search is completed, the search distribution $\pi$ is computed using the simulation visit counts (with search temperature $\tau$):
$$\pi_a = \frac{N(s, a)^{1/\tau}}{\sum_{b} N(s, b)^{1/\tau}}$$
The model selects the actual move by sampling from $\pi$ in the opening plies ($\|\text{history}\| < 30$, $\tau = 1.0$), or taking the maximum visit count in the endgame plies ($\tau \to 0$).

###1.4. Combined Dual-Head Loss Function
The network parameters $\theta$ are trained using a joint dual loss function, combining value prediction accuracy (MSE) and policy search prediction accuracy (Cross-Entropy):
$$\mathcal{L}(\theta) = (z - v)^2 - \pi^{\top} \log \mathbf{p} + c \|\theta\|^2$$
where:
*   $(z - v)^2$ is the Mean Squared Error (MSE) of the predicted value $v$ against the actual game outcome $z \in \{-1, 1\}$ (or a draw penalty of $-0.9$ to force decisive play).
*   $-\pi^{\top} \log \mathbf{p}$ is the Cross-Entropy loss between the network's raw policy probability output $\mathbf{p}$ and MCTS search target $\pi$.
*   $c \|\theta\|^2$ is the L2 weight regularization parameter.

---

##2. Model Architecture (AlphaZeroNet V1)
This notebook uses the exact `AlphaZeroNetV1` architecture matching the baseline models, consisting of three main parts:
1.  **Stem Block:** A $3 \times 3$ convolutional layer with 192 filters, Batch Normalization, and a ReLU activation.
2.  **Residual Tower (10 ResBlocks):** 10 consecutive residual convolutional blocks. Each ResBlock passes the features through two $3 \times 3$ convolutions (192 channels, Batch Normalization, ReLU) and adds a skip connection to bypass the layers. This eliminates vanishing gradients and allows the network to learn rich spatial relations across the board.
3.  **Dual Heads:**
    *   **Policy Head (Actor):** A $1 \times 1$ conv (2 filters) followed by BatchNorm, ReLU, and a Fully Connected layer outputting a vector of size **4,272**. Each element corresponds to a logit representing the probability of selecting that move out of all possible UCI moves.
    *   **Value Head (Critic):** A $1 \times 1$ conv (1 filter) followed by BatchNorm, ReLU, a Fully Connected layer (flattened to 64 -> 256), a ReLU, and a final FC layer with a `tanh` activation outputting a single scalar $v \in [-1, 1]$.

## 1. Install Dependencies
We install the `python-chess` library for complete, bug-free move validation and board rules representation.

In [ ]:
!pip install python-chess --quiet

## 2. Imports and Environment Setup
We import core PyTorch, NumPy, and chess libraries and seed the environments to ensure reproducible behavior.

In [ ]:
import os
import time
import math
import random
from collections import deque
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import chess

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
    torch.backends.cudnn.benchmark = True
import h5py
import glob


## 3. Chess Board Encoder & Action Space mapping
This class encodes any `chess.Board` state into an `(18, 8, 8)` float32 tensor and registers the 4,272 possible UCI chess moves to coordinate indices.

In [ ]:
FILES = "abcdefgh"
RANKS = "12345678"
PROMOTIONS = ['q', 'r', 'n', 'b']

# Pre-calculate action space mapping (UCI move -> index 0-4271)
ALL_UCI_MOVES = []
for from_file in FILES:
    for from_rank in RANKS:
        for to_file in FILES:
            for to_rank in RANKS:
                ALL_UCI_MOVES.append(f"{from_file}{from_rank}{to_file}{to_rank}")

for from_file in FILES:
    for to_file in FILES:
        if abs(FILES.index(from_file) - FILES.index(to_file)) <= 1:
            for p in PROMOTIONS:
                ALL_UCI_MOVES.append(f"{from_file}7{to_file}8{p}")
                ALL_UCI_MOVES.append(f"{from_file}2{to_file}1{p}")

ALL_UCI_MOVES = sorted(list(set(ALL_UCI_MOVES)))
MOVE_TO_INDEX = {move: i for i, move in enumerate(ALL_UCI_MOVES)}
INDEX_TO_MOVE = {i: move for move, i in MOVE_TO_INDEX.items()}
NUM_MOVES = len(ALL_UCI_MOVES)

class Encoder:
    """Encodes standard chess.Board instances into highly efficient spatial state representations."""
    def __init__(self):
        self.num_channels = 18

    def board_to_tensor(self, board: chess.Board) -> np.ndarray:
        tensor = np.zeros((18, 8, 8), dtype=np.float32)

        # 1. Piece channels
        for square in chess.SQUARES:
            piece = board.piece_at(square)
            if piece:
                color = int(piece.color)
                piece_type = piece.piece_type - 1
                channel = piece_type if color == 1 else piece_type + 6
                rank, file = chess.square_rank(square), chess.square_file(square)
                tensor[channel, rank, file] = 1.0

        # 2. Turn channel
        if board.turn == chess.WHITE:
            tensor[12, :, :] = 1.0

        # 3. Castling rights channels
        if board.has_kingside_castling_rights(chess.WHITE): tensor[13, :, :] = 1.0
        if board.has_queenside_castling_rights(chess.WHITE): tensor[14, :, :] = 1.0
        if board.has_kingside_castling_rights(chess.BLACK): tensor[15, :, :] = 1.0
        if board.has_queenside_castling_rights(chess.BLACK): tensor[16, :, :] = 1.0

        # 4. En-passant channel
        if board.ep_square is not None:
            rank, file = chess.square_rank(board.ep_square), chess.square_file(board.ep_square)
            tensor[17, rank, file] = 1.0

        return tensor

    def fen_to_tensor(self, fen: str) -> np.ndarray:
        board = chess.Board(fen)
        return self.board_to_tensor(board)

    def move_to_index(self, move: chess.Move) -> int:
        uci = move.uci()
        return MOVE_TO_INDEX.get(uci, 0)

    def move_str_to_index(self, uci: str) -> int:
        return MOVE_TO_INDEX.get(uci, 0)

    def index_to_move(self, index: int) -> chess.Move:
        uci = INDEX_TO_MOVE[index]
        return chess.Move.from_uci(uci)


## 4. AlphaZeroNet V1 Architecture
This is the exact original neural architecture consisting of 10 residual blocks and 192 channels, equipped with dual heads for policy logits and state evaluation.

In [ ]:
class ResBlockV1(nn.Module):
    """Standard residual block with batch normalization and skip-connections."""
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, kernel_size=3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + residual
        return F.relu(out)

class AlphaZeroNetV1(nn.Module):
    """
    10 ResBlocks, 192 channels.
    Policy Head: 2-channel conv -> FC(4272)
    Value Head : 1-channel conv -> FC(256) -> FC(1) -> tanh
    """
    def __init__(self, num_blocks: int = 10, num_channels: int = 192,
                 num_input_channels: int = 18, num_moves: int = 4272):
        super().__init__()

        # Stem layer
        self.initial_conv = nn.Sequential(
            nn.Conv2d(num_input_channels, num_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(num_channels),
            nn.ReLU(inplace=True)
        )

        # Residual tower
        self.res_blocks = nn.ModuleList([
            ResBlockV1(num_channels) for _ in range(num_blocks)
        ])

        # Policy Head (Actor)
        self.policy_conv = nn.Conv2d(num_channels, 2, kernel_size=1, bias=False)
        self.policy_bn   = nn.BatchNorm2d(2)
        self.policy_fc   = nn.Linear(2 * 8 * 8, num_moves)

        # Value Head (Critic)
        self.value_conv  = nn.Conv2d(num_channels, 1, kernel_size=1, bias=False)
        self.value_bn    = nn.BatchNorm2d(1)
        self.value_fc1   = nn.Linear(1 * 8 * 8, 256)
        self.value_fc2   = nn.Linear(256, 1)

    def forward(self, x: torch.Tensor):
        x = self.initial_conv(x)
        for block in self.res_blocks:
            x = block(x)

        # Policy predictions
        p = F.relu(self.policy_bn(self.policy_conv(x)))
        p = p.reshape(p.size(0), -1)
        policy_logits = self.policy_fc(p)

        # Value predictions
        v = F.relu(self.value_bn(self.value_conv(x)))
        v = v.reshape(v.size(0), -1)
        v = F.relu(self.value_fc1(v))
        value = torch.tanh(self.value_fc2(v))

        return policy_logits, value

## 5. Supervised Pre-training (Finetuning) on Grandmaster Moves

Before starting the computationally heavy self-play reinforcement learning loop, it is a standard practice (e.g. in the original AlphaGo) to initialize or **pre-train** the neural network on strong human moves. This is known as **Supervised Pre-training (SL)**.

### 5.1. Why Supervised Learning?
*   **Overcoming Random Play:** Tabula-rasa RL starts with a completely random network, wasting millions of steps just learning that pawns cannot jump sideways or how checkmates work.
*   **Seeding Strong Priors:** Pre-training on Grandmaster (GM) games allows the model's **Policy Head** to predict strong moves with high accuracy (~30-45%) from Day 1, which immediately guides MCTS searches down highly promising paths.
*   **Balanced Value Estimation:** Pre-training the **Value Head** on game outcomes ensures MCTS nodes are evaluated correctly, reducing random fluctuations.

### 5.2. Dataset & Streaming Architecture
We train the model using a large preprocessed dataset of GM moves compiled into high-speed binary **HDF5 (.h5) files**. The data is loaded dynamically using a custom PyTorch `Dataset` that indexes file ranges and streams values without exhausting RAM.

In [ ]:
class ChessDataset(torch.utils.data.Dataset):
    """
    Highly efficient PyTorch Dataset streaming preprocessed HDF5 files containing GM games.
    """
    def __init__(self, data_dir: str, max_files: int = None, limit_positions: int = None):
        self.data_dir = data_dir
        self.h5_files = sorted(glob.glob(os.path.join(data_dir, '*.h5')))
        if max_files is not None:
            self.h5_files = self.h5_files[:max_files]
            
        self.encoder = Encoder()
        self.limit_positions = limit_positions
        
        self.file_sizes = []
        self.cumulative_sizes = []
        self.total_size = 0
        
        print(f"🔍 Indexing {len(self.h5_files)} HDF5 files in '{data_dir}'...")
        for f in self.h5_files:
            try:
                with h5py.File(f, 'r') as h5f:
                    if 'fen' in h5f:
                        size = len(h5f['fen'])
                        if self.limit_positions is not None and self.total_size + size >= self.limit_positions:
                            size = self.limit_positions - self.total_size
                            self.file_sizes.append(size)
                            self.total_size += size
                            self.cumulative_sizes.append(self.total_size)
                            break
                            
                        self.file_sizes.append(size)
                        self.total_size += size
                        self.cumulative_sizes.append(self.total_size)
            except Exception as e:
                print(f"⚠️ Error reading file {f}: {e}")
                self.file_sizes.append(0)
                self.cumulative_sizes.append(self.total_size)
                
        print(f"✅ Indexing complete. Found {self.total_size} total GM board positions.")
        self.current_h5 = None
        self.current_f_idx = -1

    def __len__(self):
        return self.total_size

    def __getitem__(self, idx):
        f_idx = np.searchsorted(self.cumulative_sizes, idx, side='right')
        if f_idx == 0:
            local_idx = idx
        else:
            local_idx = idx - self.cumulative_sizes[f_idx - 1]
            
        if self.current_f_idx != f_idx or self.current_h5 is None:
            if self.current_h5 is not None:
                self.current_h5.close()
            self.current_h5 = h5py.File(self.h5_files[f_idx], 'r')
            self.current_f_idx = f_idx
            
        fen_bytes = self.current_h5['fen'][local_idx]
        move_bytes = self.current_h5['move'][local_idx]
        value = self.current_h5['value'][local_idx]
        
        fen = fen_bytes.decode('utf-8')
        move_str = move_bytes.decode('utf-8')
        
        tensor = self.encoder.fen_to_tensor(fen)
        move_idx = self.encoder.move_str_to_index(move_str)
        
        return torch.from_numpy(tensor), torch.tensor(move_idx, dtype=torch.long), torch.tensor(value, dtype=torch.float32)

    def __del__(self):
        if self.current_h5 is not None:
            self.current_h5.close()

def get_dataloader(data_dir: str, batch_size: int, num_workers: int = 4, max_files: int = None, limit_positions: int = None):
    dataset = ChessDataset(data_dir, max_files, limit_positions)
    loader = torch.utils.data.DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers,
        pin_memory=True
    )
    return loader

In [ ]:
def train_supervised(model, data_dir, epochs=5, batch_size=4096, num_workers=4, limit_positions=6000000, lr=5e-5):
    """
    Supervised Finetuning loop to pre-train policy/value heads on grandmaster datasets.
    """
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"🚀 Starting Supervised Finetuning on: {device}...")
    
    if not os.path.exists(data_dir) or not glob.glob(os.path.join(data_dir, '*.h5')):
        print(f"❌ ERROR: Data directory '{data_dir}' not found or contains no .h5 files.")
        print("👉 Make sure to mount your grandmaster HDF5 dataset to this path on Kaggle.")
        return False
        
    model = model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)    
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
    
    policy_criterion = nn.CrossEntropyLoss()
    value_criterion = nn.MSELoss()
    
    dataloader = get_dataloader(
        data_dir, 
        batch_size, 
        num_workers=num_workers,
        limit_positions=limit_positions
    )
    
    os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
    best_loss = float('inf')
    best_ckpt_path = '/kaggle/working/checkpoints/alphazero_pretrain_best.pth'
    
    for epoch in range(epochs):
        current_lr = scheduler.get_last_lr()[0]
        print(f"\n--- Supervised Epoch {epoch+1}/{epochs} (Learning Rate = {current_lr:.6f}) ---")
        
        model.train()
        total_p_loss = 0.0
        total_v_loss = 0.0
        total_loss = 0.0
        start_time = time.time()
        
        for batch_idx, (tensors, moves, values) in enumerate(dataloader):
            tensors, moves, values = tensors.to(device), moves.to(device), values.to(device)
            
            # Forward Pass
            policy_logits, value_out = model(tensors)
            value_out = value_out.squeeze(1)
            
            # Policy Loss (Cross-Entropy matching human Grandmaster move choice)
            loss_p = policy_criterion(policy_logits, moves)
            # Value Loss (MSE matching game result)
            loss_v = value_criterion(value_out, values)
            
            loss = loss_p + loss_v
            
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_p_loss += loss_p.item()
            total_v_loss += loss_v.item()
            total_loss += loss.item()
            
            if batch_idx % 100 == 0:
                print(f"  Batch {batch_idx:4d}/{len(dataloader)} | "
                      f"P_Loss: {loss_p.item():.4f} | V_Loss: {loss_v.item():.4f} | "
                      f"Total Loss: {loss.item():.4f}")
                      
        avg_loss = total_loss / len(dataloader)
        avg_p_loss = total_p_loss / len(dataloader)
        avg_v_loss = total_v_loss / len(dataloader)
        epoch_time = time.time() - start_time
        
        print(f"✨ Epoch {epoch+1} Complete | Time: {epoch_time:.1f}s | Avg Loss: {avg_loss:.4f}")
        print(f"   Policy Loss: {avg_p_loss:.4f} | Value Loss: {avg_v_loss:.4f}")
        
        scheduler.step()
        
        # Save best pre-trained model
        if avg_loss < best_loss:
            best_loss = avg_loss
            print(f"  🏆 New best pre-trained checkpoint! Saving to '{best_ckpt_path}'...")
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': best_loss,
            }, best_ckpt_path)
            
    print(f"\n🎉 Supervised pre-training finished! Best model saved to: '{best_ckpt_path}'")
    return True

### 5.3. Execute Supervised Pre-Training

Set `RUN_SUPERVISED_PRETRAINING = False` (default) to skip pre-training and train from scratch via self-play, or set it to `True` and specify your Kaggle input directory path to pre-train the model first.

*(Note: You can load the pre-trained weights directly into the reinforcement learning loop afterwards!)*

In [ ]:
RUN_SUPERVISED_PRETRAINING = False  # Set to True if GM dataset is mounted
GRANDMASTER_DATA_DIR = "/kaggle/input/chess-grandmaster-dataset/train_top_6m"

if RUN_SUPERVISED_PRETRAINING:
    # Pre-train for 5 epochs on up to 6 million GM board positions
    success = train_supervised(
        model=model, 
        data_dir=GRANDMASTER_DATA_DIR, 
        epochs=5, 
        batch_size=4096, 
        num_workers=4, 
        limit_positions=6000000, 
        lr=5e-5
    )
    if success:
        # Load the best pre-trained weights to seed self-play RL
        best_pretrain_path = "/kaggle/working/checkpoints/alphazero_pretrain_best.pth"
        checkpoint = torch.load(best_pretrain_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        print("✅ Successfully loaded GM pre-trained weights into AlphaZero model for RL!")
else:
    print("➡️ Skipping Supervised Pre-training. AlphaZero will learn tabula-rasa from pure self-play!")

## 5. MCTS Node & Batched Vectorized Self-Play
Here we define our MCTS search nodes, root Dirichlet Noise injection, draw penalization parameters, and the high-speed vectorized environment runner that plays 256 parallel self-play games.

In [ ]:
class MCTSNode:
    """Represents a search node in the MCTS tree."""
    def __init__(self, prior=0.0):
        self.P = prior
        self.N = 0
        self.W = 0.0
        self.Q = 0.0
        self.children = {}  # action -> MCTSNode

    def expand(self, legal_moves, action_probs, board=None):
        for move in legal_moves:
            idx = MOVE_TO_INDEX.get(move.uci())
            if idx is not None:
                p = action_probs[idx]
                # Mate-in-1 Biasing to accelerate checkmates discovery
                if board is not None:
                    board.push(move)
                    if board.is_checkmate():
                        p += 100.0
                    board.pop()
                self.children[move] = MCTSNode(prior=p)

    def is_leaf(self):
        return len(self.children) == 0

    def select_child(self, c_puct=2.0):
        best_uct = -float('inf')
        best_child = None
        best_action = None

        sum_N = math.sqrt(self.N + 1)
        for action, child in self.children.items():
            u = c_puct * child.P * sum_N / (1 + child.N)
            uct = child.Q + u
            if uct > best_uct:
                best_uct = uct
                best_child = child
                best_action = action
        return best_action, best_child

    def backprop(self, value):
        self.N += 1
        self.W += value
        self.Q = self.W / self.N

def apply_dirichlet_noise(node, alpha=0.3, epsilon=0.25):
    """Inject Dirichlet noise to encourage exploration of new lines at the root node."""
    actions = list(node.children.keys())
    if not actions: return
    noise = np.random.dirichlet([alpha] * len(actions))
    for i, action in enumerate(actions):
        node.children[action].P = (1 - epsilon) * node.children[action].P + epsilon * noise[i]

def get_terminal_value(board):
    """Heavy penalty for draws to force aggressive winning play."""
    res = board.result()
    if res == '1/2-1/2':
        return -0.9  # Draw is almost as bad as a loss
    return -1.0      # Loser gets -1.0

def format_z_for_history(history, result_str):
    """Z-target generator. Propagates symmetric outcomes to self-play trajectories."""
    targets = []
    for (state, pi, color_turn) in history:
        if result_str == '1/2-1/2':
            z = -0.9 # Both get penalized to discourage draws
        elif result_str == '1-0':
            z = 1.0 if color_turn == chess.WHITE else -1.0
        elif result_str == '0-1':
            z = 1.0 if color_turn == chess.BLACK else -1.0
        targets.append((state, pi, z))
    return targets

class VectorSelfPlay:
    """Handles multiple games running in parallel to speed up GPU inference throughput."""
    def __init__(self, num_games=32):
        self.num_games = num_games
        self.boards = [chess.Board() for _ in range(num_games)]
        self.roots  = [MCTSNode() for _ in range(num_games)]
        self.histories = [[] for _ in range(num_games)]
        self.encoder = Encoder()

    def step_games(self, model, device, simulations=30):
        # 1. Apply Dirichlet noise at root
        for i in range(self.num_games):
            if not self.boards[i].is_game_over() and len(self.roots[i].children) > 0:
                apply_dirichlet_noise(self.roots[i])

        # 2. Run batched MCTS simulations
        for _ in range(simulations):
            leaf_tensors = []
            leaf_infos = []  # (board_idx, moves_taken, legal_moves)
            search_paths = []

            for i in range(self.num_games):
                if self.boards[i].is_game_over():
                    continue

                b = self.boards[i]
                node = self.roots[i]
                path = [node]
                moves_taken = []

                # Tree selection phase
                while not node.is_leaf():
                    action, child = node.select_child()
                    path.append(child)
                    b.push(action)
                    moves_taken.append(action)
                    node = child

                # Check if game ended in the simulation
                is_draw = b.can_claim_draw() or b.is_repetition(2)
                if b.is_game_over() or is_draw:
                    val = -0.9 if is_draw else get_terminal_value(b)
                    # Backpropagation
                    for n in reversed(path):
                        val = -val * 0.99
                        n.backprop(val)
                else:
                    leaf_tensors.append(self.encoder.board_to_tensor(b))
                    leaf_infos.append((i, moves_taken.copy(), list(b.legal_moves)))
                    search_paths.append(path)

                # Revert board state
                for _ in moves_taken:
                    b.pop()

            if not leaf_tensors:
                break

            # Batch inference on GPU for leaf nodes
            batch_t = torch.tensor(np.array(leaf_tensors), dtype=torch.float32).to(device)
            with torch.no_grad():
                logits, values = model(batch_t)
                probs = F.softmax(logits, dim=1).cpu().numpy()
                values = values.squeeze(1).cpu().numpy()

            # Expansion and backpropagation
            for k, (board_idx, moves_taken, legal_moves) in enumerate(leaf_infos):
                b = self.boards[board_idx]
                for action in moves_taken:
                    b.push(action)

                node = search_paths[k][-1]
                node.expand(legal_moves, probs[k], b)

                for _ in moves_taken:
                    b.pop()

                val = values[k]
                for n in reversed(search_paths[k]):
                    val = -val * 0.99
                    n.backprop(val)

        # 3. Action Selection
        finished_games = []
        for i in range(self.num_games):
            if self.boards[i].is_game_over():
                continue

            root = self.roots[i]
            pi = np.zeros(NUM_MOVES, dtype=np.float32)
            total_n = sum(c.N for c in root.children.values())

            if total_n == 0:
                actions = list(self.boards[i].legal_moves)
                action = random.choice(actions)
                pi[MOVE_TO_INDEX[action.uci()]] = 1.0
            else:
                actions = []
                visits = []
                for a, c in root.children.items():
                    actions.append(a)
                    visits.append(c.N)
                    pi[MOVE_TO_INDEX[a.uci()]] = c.N / total_n

                #opening exploration plies (e.g. first 30 moves)
                if len(self.histories[i]) < 30:
                    action = random.choices(actions, weights=visits, k=1)[0]
                else:
                    action = actions[np.argmax(visits)]

            state_tensor = self.encoder.board_to_tensor(self.boards[i])
            self.histories[i].append((state_tensor, pi, self.boards[i].turn))
            self.boards[i].push(action)

            # Advance search tree root
            if action in root.children:
                self.roots[i] = root.children[action]
            else:
                self.roots[i] = MCTSNode()

            # Reset if game over or exceeded max plies
            if self.boards[i].is_game_over() or len(self.histories[i]) > 200:
                res = self.boards[i].result() if self.boards[i].is_game_over() else '1/2-1/2'
                finished_games.extend(format_z_for_history(self.histories[i], res))

                self.boards[i] = chess.Board()
                self.roots[i] = MCTSNode()
                self.histories[i] = []

        return finished_games

## 6. Training Configuration & Hyperparameters
We instantiate our model, configure optimizer parameters (AdamW with Cosine Annealing scheduler), declare checkpoints directory, and initialize the parallel environment.

In [ ]:
# GPU Acceleration check
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Executing training on: {device}")

# Checkpoint Directories
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)
latest_checkpoint_path = '/kaggle/working/checkpoints/alphazero_latest.pth'
best_checkpoint_path = '/kaggle/working/checkpoints/alphazero_best.pth'

# Hyperparameters
NUM_PARALLEL_GAMES = 32      # Matches resource bounds on T4 GPU
SIMULATIONS_PER_MOVE = 30    # MCTS simulation rollout counts
REPLAY_BUFFER_CAPACITY = 50000
BATCH_SIZE = 512
GAMES_BEFORE_TRAIN = 1000    # Minimum moves inside buffer before updating net
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
EPOCHS_PER_ITERATION = 3     # Training epochs over sampled batch

# Model Creation
model = AlphaZeroNetV1(num_blocks=10, num_channels=192).to(device)

# Load pre-trained weights if they exist from the Supervised learning phase
best_pretrain_path = '/kaggle/working/checkpoints/alphazero_pretrain_best.pth'
if os.path.exists(best_pretrain_path):
    checkpoint = torch.load(best_pretrain_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("✅ Seeded Reinforcement Learning with pre-trained Grandmaster moves weights!")
else:
    print("➡️ No pre-trained weights found. Starting Reinforcement Learning from scratch.")
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

# Warmup Root nodes
env = VectorSelfPlay(num_games=NUM_PARALLEL_GAMES)
for i in range(NUM_PARALLEL_GAMES):
    legal = list(env.boards[i].legal_moves)
    tensor = torch.tensor(env.encoder.board_to_tensor(env.boards[i])).unsqueeze(0).to(device)
    with torch.no_grad():
        pol, _ = model(tensor)
    pol = F.softmax(pol, dim=1).cpu().numpy()[0]
    env.roots[i].expand(legal, pol, env.boards[i])

replay_buffer = deque(maxlen=REPLAY_BUFFER_CAPACITY)


## 7. AlphaZero Training Loop
Executes our iterative training schedule. The loop generates high-quality self-play games, accumulates experiences, and updates the neural network on both actor (Policy) and critic (Value) targets.

In [ ]:
# Track training metrics
history_iterations = []
history_policy_loss = []
history_value_loss = []
history_total_loss = []

start_time = time.time()
MAX_ITERATIONS = 500  # Expand according to your training budget
best_loss = float('inf')

print("Populating Replay Buffer with initial MCTS self-play games...")
while len(replay_buffer) < GAMES_BEFORE_TRAIN:
    finished = env.step_games(model, device, simulations=SIMULATIONS_PER_MOVE)
    replay_buffer.extend(finished)
    print(f"  Buffer Fill: {len(replay_buffer)}/{GAMES_BEFORE_TRAIN}", end='\r')
print(f"\nBuffer populated! Starting core optimization loop...")

for iteration in range(1, MAX_ITERATIONS + 1):
    model.eval()
    new_data = []

    # 1. Play steps to collect fresh data
    while len(new_data) < 200:
        finished = env.step_games(model, device, simulations=SIMULATIONS_PER_MOVE)
        new_data.extend(finished)

    replay_buffer.extend(new_data)

    # 2. Gradient Updates
    model.train()
    train_samples = random.sample(replay_buffer, min(len(replay_buffer), BATCH_SIZE * 4))

    total_p_loss = 0.0
    total_v_loss = 0.0
    total_loss = 0.0
    steps = 0

    for epoch in range(EPOCHS_PER_ITERATION):
        random.shuffle(train_samples)
        batches = [train_samples[i:i+BATCH_SIZE] for i in range(0, len(train_samples), BATCH_SIZE)]

        for batch in batches:
            if len(batch) < 32: continue

            states = torch.tensor(np.array([x[0] for x in batch]), dtype=torch.float32).to(device)
            pi_ts  = torch.tensor(np.array([x[1] for x in batch]), dtype=torch.float32).to(device)
            z_ts   = torch.tensor(np.array([x[2] for x in batch]), dtype=torch.float32).to(device)

            # Forward Pass
            p_logits, v_out = model(states)
            v_out = v_out.squeeze(1)

            # 2a. Actor Policy Loss (CrossEntropy against search probabilities)
            loss_p = -(pi_ts * F.log_softmax(p_logits, dim=1)).sum(dim=1).mean()

            # 2b. Critic Value Loss (MSE against game terminal result z)
            loss_v = F.mse_loss(v_out, z_ts)

            # Combined Loss
            loss = loss_p + loss_v

            # Backpropagation
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            total_p_loss += loss_p.item()
            total_v_loss += loss_v.item()
            total_loss += loss.item()
            steps += 1

    avg_p_loss = total_p_loss / max(1, steps)
    avg_v_loss = total_v_loss / max(1, steps)
    avg_loss   = total_loss / max(1, steps)

    # Log history metrics
    history_iterations.append(iteration)
    history_policy_loss.append(avg_p_loss)
    history_value_loss.append(avg_v_loss)
    history_total_loss.append(avg_loss)

    # Print Progress
    if iteration % 10 == 0 or iteration == 1:
        elapsed = (time.time() - start_time) / 60
        print(f"Iteration {iteration:4d} | Policy Loss: {avg_p_loss:.4f} | Value Loss: {avg_v_loss:.4f} | "
              f"Total Loss: {avg_loss:.4f} | Buffer: {len(replay_buffer):5d} | Elapsed: {elapsed:.1f}m")

        # Save latest checkpoint
        torch.save({
            'iteration': iteration,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss
        }, latest_checkpoint_path)

        # Save best checkpoint
        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                'iteration': iteration,
                'model_state_dict': model.state_dict(),
                'loss': best_loss
            }, best_checkpoint_path)
            print(f"  => Saved new BEST model with Loss {best_loss:.4f} to '{best_checkpoint_path}'")

## 8. Visualize Training Progress
Plots policy loss, value loss, and total joint loss across iterations.

In [ ]:
plt.figure(figsize=(15, 5))

# Policy Loss
plt.subplot(1, 3, 1)
plt.plot(history_iterations, history_policy_loss, label='Policy Loss', color='mediumvioletred')
plt.title('Actor (Policy) Loss')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()

# Value Loss
plt.subplot(1, 3, 2)
plt.plot(history_iterations, history_value_loss, label='Value Loss', color='dodgerblue')
plt.title('Critic (Value) Loss')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()

# Total Joint Loss
plt.subplot(1, 3, 3)
plt.plot(history_iterations, history_total_loss, label='Total Loss', color='forestgreen')
plt.title('Total Joint Loss')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.savefig('/kaggle/working/alphazero_training_curves.png')
plt.show()

print("Training process finished! Output models and visualization plots saved in '/kaggle/working/'.")